# Inner Product Spaces

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 01.01–01.10, 05.01–05.07, 06.01–06.07, 07.01–07.05 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/08_Advanced_Linear_Algebra/01_inner_product_spaces.ipynb)

---

## 🎯 Learning Objective

Generalize the dot product to abstract inner products, and learn orthogonality and the projection theorem in Hilbert spaces.

---

## 📐 Theory

[Notebook 01.01](../01_Linear_Algebra/01_vectors_and_spaces.ipynb) introduced the dot product
$\mathbf{u}\cdot\mathbf{v}=\sum_i u_i v_i$ as a way to measure angle and length together. This
notebook asks: what if we replace the dot product with *any* rule that behaves like it?
Generalizing unifies some of the most-used tricks in ML — kernel methods, attention, Gram
matrices — under one piece of theory.

### The inner product axioms

An **inner product** on a (real) vector space $V$ is any function $\langle\cdot,\cdot\rangle:
V\times V\to\mathbb{R}$ satisfying, for all $\mathbf{u},\mathbf{v},\mathbf{w}\in V$ and scalar $c$:

1. **Symmetry**: $\langle\mathbf{u},\mathbf{v}\rangle = \langle\mathbf{v},\mathbf{u}\rangle$
2. **Linearity**: $\langle c\mathbf{u}+\mathbf{v},\mathbf{w}\rangle = c\langle\mathbf{u},\mathbf{w}\rangle+\langle\mathbf{v},\mathbf{w}\rangle$
3. **Positive-definiteness**: $\langle\mathbf{v},\mathbf{v}\rangle \ge 0$, with equality iff $\mathbf{v}=\mathbf{0}$

The dot product is *one* inner product on $\mathbb{R}^n$, but not the only one — and the space
itself need not be $\mathbb{R}^n$. A space equipped with an inner product is an **inner product
space**, and every inner product automatically defines a norm
$\|\mathbf{v}\| = \sqrt{\langle\mathbf{v},\mathbf{v}\rangle}$ and an angle via
$\cos\theta = \langle\mathbf{u},\mathbf{v}\rangle/(\|\mathbf{u}\|\|\mathbf{v}\|)$ — "length" and
"angle" transplant to any setting with a valid $\langle\cdot,\cdot\rangle$.

Two examples that matter downstream: a **weighted inner product**
$\langle\mathbf{u},\mathbf{v}\rangle_W = \mathbf{u}^\top W\mathbf{v}$ for any symmetric
positive-definite $W$ (recall positive-definiteness from
[01.06](../01_Linear_Algebra/06_eigenvalues_eigenvectors.ipynb); $W=I$ recovers the dot
product), and a **function-space inner product** $\langle f,g\rangle = \int_a^b f(x)g(x)\,dx$
for functions $f,g$ on $[a,b]$. Vectors don't have to be lists of numbers — polynomials,
signals, and entire feature maps can be treated as vectors in their own inner product space.
A space where such integrals stay finite and the space is "complete" is a **Hilbert space** —
the natural infinite-dimensional generalization of $\mathbb{R}^n$ where kernel methods live
([11.02](../11_Functional_Analysis/02_kernel_methods_rkhs.ipynb) formalizes this).

### Orthogonality and the Gram matrix

Two vectors are **orthogonal** ($\mathbf{u}\perp\mathbf{v}$) if $\langle\mathbf{u},\mathbf{v}\rangle=0$.
A set is **orthonormal** if every pair is orthogonal and every vector has unit norm — the
best-behaved possible basis, since $\{\mathbf{e}_i\}$ orthonormal gives
$\mathbf{v}=\sum_i \langle\mathbf{v},\mathbf{e}_i\rangle\mathbf{e}_i$ directly.

Given $k$ vectors $\mathbf{v}_1,\dots,\mathbf{v}_k$, the **Gram matrix** collects every pairwise
inner product: $G_{ij} = \langle \mathbf{v}_i, \mathbf{v}_j\rangle$. $G$ is always symmetric and
positive-*semi*-definite (same reason a covariance matrix is — both are "matrices of pairwise
inner products"), and $\text{rank}(G)$ equals the number of *linearly independent* vectors among
the $\mathbf{v}_i$ — so $G$ compresses "how do these $k$ objects relate" without needing their
raw coordinates at all. That property — only needing inner products, never the vectors
themselves — is exactly the **kernel trick**: replace $\langle\mathbf{v}_i,\mathbf{v}_j\rangle$
with a cheap function $k(\mathbf{v}_i,\mathbf{v}_j)$ that provably equals an inner product in
some (possibly huge) feature space, and every Gram-matrix-based algorithm keeps working without
ever forming that feature space.

### The projection theorem

Given a subspace $S\subseteq V$ and a point $\mathbf{b}\in V$, the **projection theorem** says
the closest point in $S$ to $\mathbf{b}$ is unique and is characterized by one condition: the
*residual* between $\mathbf{b}$ and its projection is orthogonal to all of $S$:

$$\hat{\mathbf{b}} = \arg\min_{\mathbf{s}\in S}\|\mathbf{b}-\mathbf{s}\| \quad\Longleftrightarrow\quad (\mathbf{b}-\hat{\mathbf{b}}) \perp \mathbf{s} \ \ \forall \mathbf{s}\in S$$

Geometrically: drop a perpendicular from $\mathbf{b}$ onto $S$; nothing else in $S$ is closer.
Specialized to $S=\text{range}(A)$, this *is* the derivation of the normal equations from
[01.04](../01_Linear_Algebra/04_systems_of_equations.ipynb): least-squares regression projects
$\mathbf{b}$ onto the column space of $A$, and "residual $\perp S$" becomes
$A^\top(\mathbf{b}-A\mathbf{x}^\star)=\mathbf{0}$, i.e. $A^\top A\mathbf{x}^\star = A^\top\mathbf{b}$
— reached geometrically below, rather than by setting a calculus gradient to zero.

---

In [ ]:
# Setup
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

The projection theorem says the closest point in a line (a 1D subspace) to some external point
is found by dropping a perpendicular. Plotting the point, the line, and the dropped
perpendicular makes "closest point = orthogonal residual" a picture instead of an abstract
claim — and the least-squares regression line is exactly the same picture one dimension up.

In [ ]:
# Projection theorem, made visual: the closest point on a line to an external point b
# is where the residual (dashed) meets the line at a right angle.
rng = np.random.default_rng(0)
d = np.array([2.0, 1.0])     # a direction spanning the 1D subspace S = span(d)
b = np.array([1.0, 3.5])     # external point, not on S

t_star = (b @ d) / (d @ d)   # minimizes ||b - t*d||^2; derived from the orthogonality condition below
proj = t_star * d
residual = b - proj

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

t_line = np.linspace(-0.5, 2.5, 100)
line_pts = np.outer(t_line, d)
axes[0].plot(line_pts[:, 0], line_pts[:, 1], color="#4C72B0", label="subspace S = span(d)")
axes[0].scatter(*b, color="#C44E52", zorder=5, s=60, label="b (external point)")
axes[0].scatter(*proj, color="#55A868", zorder=5, s=60, label="projection $\\hat{b}$ (closest point in S)")
axes[0].plot([b[0], proj[0]], [b[1], proj[1]], "k--", lw=1.3, label="residual $b-\\hat{b}$")
# small right-angle marker where residual meets the line
perp_dir = residual / np.linalg.norm(residual)
along_dir = d / np.linalg.norm(d)
corner = proj + 0.15 * along_dir
axes[0].plot([proj[0] + 0.15 * along_dir[0], corner[0] + 0.15 * perp_dir[0] * 0,
              proj[0] + 0.15 * along_dir[0] + 0.15 * perp_dir[0]],
             [proj[1] + 0.15 * along_dir[1], corner[1], proj[1] + 0.15 * along_dir[1] + 0.15 * perp_dir[1]],
             color="gray", lw=1)
axes[0].set_xlim(-1, 4); axes[0].set_ylim(-1, 4.5); axes[0].set_aspect("equal")
axes[0].set_title("Projection theorem: residual $\\perp$ subspace")
axes[0].legend(fontsize=8, loc="upper left")

# Right panel: sweep every OTHER candidate point on the line and confirm t_star truly
# minimizes distance to b -- ties the geometric picture to the optimization claim directly.
t_sweep = np.linspace(-0.5, 2.5, 200)
distances = np.linalg.norm(b[None, :] - t_sweep[:, None] * d[None, :], axis=1)
axes[1].plot(t_sweep, distances, color="#4C72B0")
axes[1].axvline(t_star, color="#55A868", ls="--", label=f"$t^\\star$={t_star:.3f} (from orthogonality)")
axes[1].scatter([t_star], [np.linalg.norm(residual)], color="#55A868", zorder=5)
axes[1].set_xlabel("t  (position along the line, point = t*d)")
axes[1].set_ylabel("$\\|b - t\\,d\\|$")
axes[1].set_title("Sweeping every point on S confirms $t^\\star$ is the minimum")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Orthogonality check: (b - proj) . d = {residual @ d:.2e}  (zero, as the projection theorem requires)")
print(f"t* = {t_star:.4f}, minimum distance = {np.linalg.norm(residual):.4f}")

## 🐍 Implementation from Scratch

We build a general `inner_product`/`gram_matrix` pair that accepts *any* symmetric
positive-definite weighting (not just the plain dot product), confirm the Gram matrix's rank
tracks linear dependence exactly, implement Gram-Schmidt orthogonalization against a general
inner product, and re-derive the least-squares solution directly from the projection theorem's
orthogonality condition rather than from calculus.

In [ ]:
# A general inner product (dot product is the special case W=I), its Gram matrix, and
# Gram-Schmidt built to work against ANY valid inner product -- not just the dot product.
def inner_product(u, v, W=None):
    """<u, v>_W = u^T W v. W=None means the plain dot product (W=I)."""
    return float(u @ v) if W is None else float(u @ W @ v)

def gram_matrix(vectors, W=None):
    """G_ij = <v_i, v_j>. Symmetric, PSD, and rank(G) = number of linearly independent v_i."""
    k = len(vectors)
    G = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            G[i, j] = inner_product(vectors[i], vectors[j], W)
    return G

def gram_schmidt(vectors, W=None):
    """Orthonormalize against a general inner product: subtract off each already-built
    direction's component, then normalize what's left."""
    Q = []
    for v in vectors:
        w = v.astype(float).copy()
        for q in Q:
            w = w - inner_product(w, q, W) * q
        Q.append(w / np.sqrt(inner_product(w, w, W)))
    return Q

rng = np.random.default_rng(0)
v1, v2, v3 = rng.standard_normal(5), rng.standard_normal(5), rng.standard_normal(5)
v4 = 2.0 * v1 - 0.5 * v2   # LINEARLY DEPENDENT on v1, v2 by construction
vectors = [v1, v2, v3, v4]

G = gram_matrix(vectors)
print("rank(G) =", np.linalg.matrix_rank(G), "-- 4 vectors in, but v4 is dependent, so rank is 3")
print("smallest eigenvalue:", np.round(np.linalg.eigvalsh(G)[0], 10), "(exactly 0, true degeneracy)")

# The SAME 4 vectors under a WEIGHTED inner product <u,v>_W = u^T W v, W symmetric positive-definite
B = rng.standard_normal((5, 5))
W = B @ B.T + 5 * np.eye(5)   # guaranteed SPD (see 01.06 for why B^T B is PSD)
print("Weighted Gram matrix rank:", np.linalg.matrix_rank(gram_matrix(vectors, W)),
      "-- still 3: rank-deficiency is a property of the VECTORS, not the inner product chosen.")

# Gram-Schmidt against the plain dot product, cross-checked with np.linalg.qr
Q_mat = np.stack(gram_schmidt([v1, v2, v3]), axis=1)
Q_np, _ = np.linalg.qr(np.stack([v1, v2, v3], axis=1))
print("\nGram-Schmidt result orthonormal (Q^T Q = I):", np.allclose(Q_mat.T @ Q_mat, np.eye(3)))
# Columns can differ by sign between implementations; comparing the projection matrix onto
# their span (sign-invariant) is the right way to confirm they agree.
print("Spans the same subspace as np.linalg.qr:", np.allclose(Q_mat @ Q_mat.T, Q_np @ Q_np.T))

# --- Projection theorem -> least squares, derived from orthogonality, not calculus ---
print("\n--- Least squares via the projection theorem's orthogonality condition ---")
m, n = 8, 3
A = rng.standard_normal((m, n))
b = rng.standard_normal(m)

# Projection theorem: the residual (b - A x*) must be orthogonal to EVERY column of A, i.e. to
# the whole subspace S=range(A). That's A^T(b - A x*)=0 exactly -- the normal equations, reached
# geometrically instead of by setting a calculus gradient to zero.
x_star = np.linalg.solve(A.T @ A, A.T @ b)
x_lstsq, *_ = np.linalg.lstsq(A, b, rcond=None)
print("x* (from orthogonality) matches np.linalg.lstsq:", np.allclose(x_star, x_lstsq))
residual = b - A @ x_star
print("A^T @ residual (should be exactly 0):", np.round(A.T @ residual, 10))

# Confirm x_star is truly OPTIMAL: 200 nearby points in the same subspace should all be worse.
best_norm = np.linalg.norm(residual)
worse_count = sum(
    np.linalg.norm(b - A @ (x_star + 0.05 * rng.standard_normal(n))) >= best_norm for _ in range(200)
)
print(f"Perturbing x* in {worse_count}/200 random directions never beats it -- a genuine minimum.")

## 🤖 Why This Matters for AI

Two of the most important tricks in modern ML are direct applications of this notebook's
theory. **The kernel trick** replaces an inner product $\langle\phi(\mathbf{x}),\phi(\mathbf{y})\rangle$
in some (possibly huge) feature space $\phi$ with a cheap function $k(\mathbf{x},\mathbf{y})$
that provably equals it — used by kernel SVMs, kernel PCA, and Gaussian processes to get
nonlinear decision boundaries at linear cost. **Attention** computes $QK^\top$
([recall 01.10](../01_Linear_Algebra/10_linear_algebra_in_transformers.ipynb)) — literally a
Gram matrix of every query against every key — and its rank tells you something real about how
distinguishable the keys are to the model.

In [ ]:
# 1) The kernel trick: a cheap kernel function EQUALS an inner product in a bigger feature space
import numpy as np
from sklearn.svm import SVC
from sklearn.datasets import make_circles

def poly_feature_map(X):
    # Explicit degree-2 polynomial features for 2D input: [1, sqrt2*x1, sqrt2*x2, x1^2, x2^2, sqrt2*x1*x2]
    x1, x2 = X[:, 0], X[:, 1]
    return np.stack([np.ones_like(x1), np.sqrt(2) * x1, np.sqrt(2) * x2,
                      x1 ** 2, x2 ** 2, np.sqrt(2) * x1 * x2], axis=1)

def poly_kernel(X, Y):
    # k(x,y) = (1 + x.y)^2 -- claims to equal poly_feature_map(x) . poly_feature_map(y), verified below
    return (1 + X @ Y.T) ** 2

X, y = make_circles(n_samples=200, noise=0.05, factor=0.4, random_state=0)   # NOT linearly separable in 2D

# Verify the kernel trick's CENTRAL claim numerically, before trusting anything downstream:
# does the cheap kernel function actually equal the explicit high-dimensional dot product?
Phi = poly_feature_map(X)
gram_diff = np.max(np.abs(Phi[:10] @ Phi[10:20].T - poly_kernel(X[:10], X[10:20])))
print("Kernel trick check -- k(x,y) vs <phi(x),phi(y)> explicitly, max abs difference:", gram_diff)

# A linear classifier can't separate concentric circles in raw 2D (no straight line works),
# but degree-2 features (or, equivalently, the poly kernel) make the classes linearly separable.
acc_raw = SVC(kernel="linear", C=10).fit(X, y).score(X, y)
acc_features = SVC(kernel="linear", C=10).fit(Phi, y).score(Phi, y)
acc_kernel = SVC(kernel="poly", degree=2, coef0=1, gamma=1, C=10).fit(X, y).score(X, y)  # never forms Phi
print(f"\nLinear SVM on raw (x1,x2):        accuracy={acc_raw:.3f}  (fails: classes aren't linearly separable)")
print(f"Linear SVM on EXPLICIT features:  accuracy={acc_features:.3f}")
print(f"Poly-kernel SVM (Phi never built): accuracy={acc_kernel:.3f}")
print("Kernel SVM matches the explicit-feature SVM without ever forming the 6D feature vectors --")
print("exactly the payoff of only needing inner products, never the vectors themselves.")

# 2) Attention's QK^T IS a Gram matrix -- rank of the key set limits how distinguishable queries can be
print("\n--- Attention scores as a Gram matrix (recall 01.10's Q, K, V) ---")
rng = np.random.default_rng(0)
seq_len, d_k = 5, 4
Q = rng.standard_normal((seq_len, d_k))

base_directions = rng.standard_normal((3, d_k))   # only 3 independent directions in key-space
K_degenerate = base_directions[[0, 1, 2, 0, 1]]    # 5 keys built from just those 3 directions (2 repeats)
K_full_rank = rng.standard_normal((seq_len, d_k))  # 5 keys, all independent

scores_degenerate = Q @ K_degenerate.T / np.sqrt(d_k)
scores_full = Q @ K_full_rank.T / np.sqrt(d_k)

print("rank(K_degenerate) =", np.linalg.matrix_rank(K_degenerate), " rank(K_full_rank) =", np.linalg.matrix_rank(K_full_rank))
print("Degenerate keys 0 and 3 are literally the same vector, so EVERY query scores them identically:")
print("  scores[:, 0] == scores[:, 3]:", np.allclose(scores_degenerate[:, 0], scores_degenerate[:, 3]))
print("  scores[:, 1] == scores[:, 4]:", np.allclose(scores_degenerate[:, 1], scores_degenerate[:, 4]))
print("With full-rank keys, no such forced ties exist:",
      "columns 0,3 equal?", np.allclose(scores_full[:, 0], scores_full[:, 3]))
print("\nA low-rank key matrix means the model CANNOT attend differently to two positions whose keys")
print("collapsed onto the same direction -- this is the same rank argument Module 08.05 uses for LoRA.")

## 🏋️ Exercises

### Warm-up
1. Verify by hand that $\langle\mathbf{u},\mathbf{v}\rangle_W = \mathbf{u}^\top W \mathbf{v}$ with
   $W=\begin{pmatrix}2&0\\0&1\end{pmatrix}$ satisfies the three inner-product axioms for
   $\mathbf{u}=(1,0)$, $\mathbf{v}=(0,1)$, $\mathbf{w}=(1,1)$ (symmetry and linearity are
   immediate; check positive-definiteness by confirming $\langle\mathbf{v},\mathbf{v}\rangle_W>0$
   for a couple of nonzero $\mathbf{v}$). Then verify with `inner_product`.

### Practice
2. Take 4 vectors in $\mathbb{R}^6$ where none are linearly dependent, build their Gram matrix
   with `gram_matrix`, and confirm it has full rank 4 with all-positive eigenvalues. Then replace
   one vector with a linear combination of two others and confirm the rank drops to 3 and the
   smallest eigenvalue becomes (numerically) zero — exactly as in the Implementation section, but
   with vectors you chose yourself.

### Challenge
3. Extend the kernel-trick cell: build a degree-3 polynomial feature map by hand (it will have
   more than 6 components — work out how many terms $(1+x_1+x_2)^3$-style expansion needs), verify
   it matches `(1 + X @ Y.T) ** 3` as a Gram matrix on a handful of random points, and then explain
   in a sentence why, as the polynomial degree grows, computing the kernel function directly
   becomes increasingly preferable to ever constructing the explicit feature map.

---

## 📚 Further Reading
- Schölkopf & Smola, *Learning with Kernels*, Ch. 2 (Kernels, feature spaces, and the kernel trick)
- Strang, *Introduction to Linear Algebra*, Ch. 4 (Orthogonality and least squares)
- Vaswani et al., ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762)

---

*Next notebook: [Matrix Calculus](02_matrix_calculus.ipynb)*